# Phase 5 — Legal Statute Identification (multi-label) with the four Phase-1 legal LMs

| Run | Phase-1 model | Technique | Notes |
|---|---|---|---|
| 1 | Arch 4 (ALiBi · SwiGLU · RMSNorm) | **LoRMA** (compulsory) | multiplicative low-rank update on every Linear; also the KD teacher |
| 2 | Arch 3 (GQA · learned pos · LayerNorm) | **Bottleneck adapters** | after every block, identity at init |
| 3 | Arch 2 (RoPE · SwiGLU · RMSNorm) | **Partial / layer-wise fine-tuning** | top 4 blocks + final norm, decaying LR |
| 4 | Arch 1 (sinusoidal · LayerNorm · GELU) | **Knowledge distillation** | Run-1 teacher → Arch-1 student with half its blocks (smaller, faster) |

In [ ]:
# P5-1. Setup + raw LSI data (Phase_5_LSI/Train/train.json, dev.json, Test/test.json)
import os, re, sys, json, glob, math, time, random, gc, zipfile, pickle, hashlib, warnings, traceback
from collections import Counter, defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore", message=".*use_reentrant.*")

P5_OUT = "/kaggle/working/lsi_out"       # predictions, metrics, submissions (keep: Save & Commit persists it)
P5_WORK = "/kaggle/working/lsi_work"     # caches (token store, frozen features)
os.makedirs(P5_OUT, exist_ok=True); os.makedirs(P5_WORK, exist_ok=True)
P5_SEED = 42

def p5_seed(s=P5_SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
p5_seed()

def _read_json_any(path):
    with open(path, encoding="utf-8") as f:
        raw = f.read()
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:                         # JSON-lines fallback
        data = [json.loads(l) for l in raw.splitlines() if l.strip()]
    if isinstance(data, dict):
        for k in ("data", "records", "examples", "items"):
            if isinstance(data.get(k), list):
                data = data[k]; break
        else:                                            # {id: record}
            data = [dict(v, id=v.get("id", k)) if isinstance(v, dict) else {"id": k, "text": v} for k, v in data.items()]
    return data

def _norm_record(r, need_labels):
    text = r.get("text", r.get("document", r.get("doc", "")))
    if isinstance(text, list):
        text = " ".join(t if isinstance(t, str) else " ".join(map(str, t)) for t in text)
    rec = {"id": str(r.get("id", r.get("case_id", r.get("doc_id")))), "text": str(text or "")}
    if need_labels:
        labs = r.get("labels", r.get("label", r.get("statutes", [])))
        if not isinstance(labs, (list, tuple)): labs = [labs]
        rec["labels"] = sorted({int(x) for x in labs})
    return rec

def _looks_like_lsi(path):
    try:
        d = _read_json_any(path)
        return len(d) > 0 and isinstance(d[0], dict) and isinstance(d[0].get("labels"), (list, tuple))
    except Exception:
        return False

def find_lsi_files():
    cands = sorted(glob.glob("/kaggle/input/**/train.json", recursive=True))
    cands = [c for c in cands if os.path.exists(os.path.join(os.path.dirname(c), "dev.json"))]
    cands.sort(key=lambda c: (not re.search(r"lsi|phase_?5", c, re.I), len(c)))
    for tr in cands:
        if _looks_like_lsi(tr):
            root = os.path.dirname(os.path.dirname(tr))          # .../Phase_5_LSI
            tests = glob.glob(os.path.join(root, "**", "test.json"), recursive=True)
            if not tests:
                tests = [t for t in glob.glob("/kaggle/input/**/test.json", recursive=True) if re.search(r"lsi|phase_?5", t, re.I)]
            if not tests:
                raise FileNotFoundError(f"found {tr} but no test.json next to it")
            return tr, os.path.join(os.path.dirname(tr), "dev.json"), tests[0]
    raise FileNotFoundError("Attach the Phase_5_LSI dataset: no train.json/dev.json with list-valued 'labels' under /kaggle/input")

LSI_TRAIN_PATH, LSI_DEV_PATH, LSI_TEST_PATH = find_lsi_files()
LSI_TRAIN = [_norm_record(r, True) for r in _read_json_any(LSI_TRAIN_PATH)]
LSI_DEV = [_norm_record(r, True) for r in _read_json_any(LSI_DEV_PATH)]
LSI_TEST = [_norm_record(r, False) for r in _read_json_any(LSI_TEST_PATH)]
assert len({r["id"] for r in LSI_TEST}) == len(LSI_TEST), "duplicate test ids"
print(f"✓ LSI data: train {len(LSI_TRAIN)} | dev {len(LSI_DEV)} | test {len(LSI_TEST)}")
print(f"  {LSI_TRAIN_PATH}\n  {LSI_DEV_PATH}\n  {LSI_TEST_PATH}")
_w = np.array([len(r["text"].split()) for r in LSI_TRAIN[:3000]])
print(f"  words/doc (train sample): median {int(np.median(_w))}, 90th pct {int(np.percentile(_w, 90))}, max {_w.max()}")
print(f"  labels/doc (train): mean {np.mean([len(r['labels']) for r in LSI_TRAIN]):.2f}")

In [ ]:
# P5-2. Phase-1 model code: re-use the Phase-4 notebook's cells 1-6 (config, checkpoint discovery, tokenizer,
#       architectures 1-4, lenient checkpoint loader) instead of copying them here.
#   * Attach phase4-allrun-v4.ipynb (as a Kaggle dataset, or the Phase-4 notebook itself via Add Input) and the
#     four Phase-1 checkpoint datasets exactly as in Phase 4.
#   * The pre-tokeniser and forward-pass variants chosen by Phase-4 cell 7 are read from Phase-4's
#     diagnostics.json if it is attached (Add Input -> Your Work -> the committed Phase-4 run); otherwise
#     cell 7 is re-run here on LSI text.
import ast
PHASE4_NOTEBOOK = None          # path to the Phase-4 .ipynb; None = search /kaggle/input and /kaggle/working
P4_BOOT_CELLS = (1, 2, 3, 4, 5, 6)
PREPROC_OVERRIDE = None         # e.g. "lower_punct" if you know Phase-4's chosen pre-tokeniser
REQUIRED = ("build_backbone", "pretokenize", "tokenizer")

def _find_phase4_nb():
    hits = glob.glob("/kaggle/input/**/*.ipynb", recursive=True) + glob.glob("/kaggle/working/**/*.ipynb", recursive=True)
    hits = [h for h in hits if "checkpoint" not in h.lower() or "phase" in h.lower()]
    hits.sort(key=lambda h: (not re.search(r"phase.?4|allrun|cjpe", h, re.I), not re.search(r"v4", h, re.I), len(h)))
    return hits[0] if hits else None

def _p4_cells(path):
    nb = json.load(open(path, encoding="utf-8"))
    out = {}
    for c in nb.get("cells", []):
        if c.get("cell_type") != "code": continue
        src = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        head = "\n".join(src.splitlines()[:5])
        m = re.search(r"^#\s*(?:CELL\s*)?(\d+)\.\s", head, re.M | re.I)
        if m and int(m.group(1)) not in out:
            out[int(m.group(1))] = src
    return out

def _sanitize(src):
    lines = []
    for l in src.splitlines():
        s = l.lstrip()
        lines.append(l[:len(l) - len(s)] + "pass  # [P5 skipped] " + s if s.startswith(("!", "%")) else l)
    return "\n".join(lines)

_SAFE_CALLS = {"dict", "list", "tuple", "set", "int", "float", "str", "range", "frozenset", "sorted", "len", "bool"}

def _heavy(node):
    """True if the statement calls anything beyond trivial constructors (lambda bodies are not evaluated)."""
    todo = [node]
    while todo:
        n = todo.pop()
        if isinstance(n, ast.Lambda): continue
        if isinstance(n, ast.Call):
            f = n.func
            name = f.id if isinstance(f, ast.Name) else (f.attr if isinstance(f, ast.Attribute) else None)
            if name not in _SAFE_CALLS: return True
        todo.extend(ast.iter_child_nodes(n))
    return False

def _defs_only(src):
    """Keep imports, function/class definitions and constant assignments of a Phase-4 cell; drop the work it does."""
    tree = ast.parse(_sanitize(src))
    keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef))
            or (isinstance(n, (ast.Assign, ast.AnnAssign)) and not _heavy(n))]
    # each statement on its own, so one assignment referring to something we skipped cannot block the rest
    return "\n".join("try:\n" + "\n".join("    " + l for l in ast.unparse(n).splitlines()) + "\nexcept NameError:\n    pass"
                     for n in keep)

def _exec_p4(n, src):
    t0 = time.time()
    try:
        exec(compile(_sanitize(src), f"<phase4 cell {n}>", "exec"), globals())
        print(f"  ✓ Phase-4 cell {n} ({time.time() - t0:.0f}s)")
        return True
    except Exception as e:
        print(f"  ⚠ Phase-4 cell {n} raised {type(e).__name__}: {str(e)[:300]}")
        return False

if PHASE4_NOTEBOOK != "skip":
    PHASE4_NOTEBOOK = PHASE4_NOTEBOOK or _find_phase4_nb()
    assert PHASE4_NOTEBOOK, "Phase-4 notebook not found: attach it, or paste Phase-4 cells 1-7 above and set PHASE4_NOTEBOOK='skip'"
    print("Importing Phase-1 model code from", PHASE4_NOTEBOOK)
    _P4 = _p4_cells(PHASE4_NOTEBOOK)
    print("  numbered cells found:", sorted(_P4))
    for n in P4_BOOT_CELLS:
        if n in _P4: _exec_p4(n, _P4[n])
        else: print(f"  ⚠ Phase-4 cell {n} not found")

if "device" not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if "free_mem" not in globals():
    def free_mem():
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

# ---- pre-tokeniser + forward-pass variants chosen in Phase 4 -------------------
def _find_p4_diag():
    for p in sorted(glob.glob("/kaggle/input/**/diagnostics.json", recursive=True)):
        try:
            d = json.load(open(p))
        except Exception:
            continue
        if "preproc" in d: return p, d
    return None, None

_DIAG_PATH, _DIAG = _find_p4_diag()
if PHASE4_NOTEBOOK != "skip" and 7 in _P4:
    if _DIAG or PREPROC_OVERRIDE:          # cell 7 holds the backbone builders: take its definitions, skip the slow grid
        _exec_p4("7 (definitions only)", _defs_only(_P4[7]))
    else:
        print("No Phase-4 diagnostics.json attached -> running Phase-4 cell 7 (pre-tokeniser / variant check) on LSI dev text")
        dev_records = [{"text": r["text"], "label": 0} for r in LSI_DEV]     # cell 7 samples text from dev_records
        _exec_p4(7, _P4[7])

if _DIAG:
    PREPROC = _DIAG["preproc"]
    if "ARCH_OPTS" in globals() and isinstance(_DIAG.get("arch_opts"), dict):
        for k, v in _DIAG["arch_opts"].items():
            ARCH_OPTS[int(k)] = dict(v)
    print(f"✓ Phase-4 diagnostics applied from {_DIAG_PATH}: PREPROC={PREPROC}, ARCH_OPTS={globals().get('ARCH_OPTS')}")
    if _DIAG.get("ppl"): print("  Phase-4 perplexities:", _DIAG["ppl"])
if PREPROC_OVERRIDE:
    PREPROC = PREPROC_OVERRIDE
if globals().get("PREPROC", "auto") == "auto":
    PREPROC = PREPROC_MODES[0] if "PREPROC_MODES" in globals() else "raw"
    print(f"⚠ Pre-tokeniser could not be determined, falling back to '{PREPROC}'. Set PREPROC_OVERRIDE.")

_missing = [n for n in REQUIRED if n not in globals()]
assert not _missing, f"Phase-1 code incomplete, missing {_missing}. Check the warnings above."
PAD_ID = 0
for _k in ("<pad>", "[PAD]", "<PAD>"):
    if hasattr(tokenizer, "stoi") and _k in tokenizer.stoi:
        PAD_ID = int(tokenizer.stoi[_k]); break
print(f"✓ Phase-1 code ready | device {device} | PREPROC={PREPROC} | PAD_ID={PAD_ID}")

In [ ]:
# P5-3. Phase-5 configuration (after the Phase-4 import, so these values win)
P5_DRY_RUN = False                 # True: small subsets + short budgets, to check the whole notebook in ~20 min
P5_RUNS = [1, 2, 3, 4]             # split across Kaggle sessions if needed: outputs persist in P5_OUT
P5_ABLATION = True                 # cheap frozen-feature ablations (context selection / pooling / loss / label correlation)

# ---- long-document representation --------------------------------------------
P5_CHUNK_LEN = int(globals().get("CHUNK_LEN", 512))   # tokens per chunk (Phase-4 value; must be <= Phase-1 context)
P5_OVERLAP = 64                    # tokens shared by consecutive chunks (sliding window)
P5_HEAD_CHUNKS, P5_TAIL_CHUNKS, P5_KEY_CHUNKS = 4, 2, 8   # candidate pool per document
P5_CAP = 6                         # chunks per document actually fed to the model
P5_SELECT = "keyword"              # 'keyword' (statute-reference scoring) | 'headtail' | 'all'; ablation may override

# ---- head / loss / training --------------------------------------------------
P5_HID = 384
P5_POOL = "lwan"                   # 'lwan' = label-wise attention over chunks | 'meanmax'
P5_LOSS = "asl"                    # 'asl' (asymmetric loss) | 'wbce' (pos-weighted BCE) | 'bce'
P5_CORR = False                    # label-correlation layer on top of the logits
P5_A_EPOCHS, P5_A_LR, P5_A_BS = 40, 1e-3, 32
P5_ABL_EPOCHS = 20
P5_FT_MIN = {1: 120, 2: 70, 3: 70, 4: 70}     # Stage-B fine-tuning minutes per run
P5_DOCS_PER_FWD, P5_ACCUM = 2, 4                # 8 documents per optimiser step; halves automatically on OOM
P5_EVAL_EVERY_MIN = 15
P5_HEAD_LR = 3e-4
P5_LORMA_R, P5_LORMA_ALPHA, P5_LORMA_LR = 8, 16.0, 5e-4
P5_ADAPTER_DIM, P5_ADAPTER_LR = 64, 5e-4
P5_LW_BLOCKS, P5_LW_TOP_LR, P5_LW_DECAY = 4, 5e-5, 0.8
P5_STUDENT_KEEP, P5_STUDENT_LR = 0.5, 5e-5      # KD student keeps this fraction of Arch-1's blocks
P5_KD_T, P5_KD_ALPHA = 2.0, 0.5
P5_MIN_POS_THR = 3                 # per-label threshold only for labels with >= this many dev positives
P5_SUB_NAME = "predictions.json"   # file name inside each submission zip

if P5_DRY_RUN:
    LSI_TRAIN, LSI_DEV, LSI_TEST = LSI_TRAIN[:300], LSI_DEV[:150], LSI_TEST[:60]
    P5_A_EPOCHS, P5_ABL_EPOCHS = 6, 3
    P5_FT_MIN = {k: 2 for k in P5_FT_MIN}; P5_EVAL_EVERY_MIN = 1
    print("⚠ DRY RUN: subsets and short budgets; submissions are written with a _DRY suffix and are NOT valid")

AMP_DTYPE_P5 = torch.float16
P5_AMP_OK = {}
print(f"✓ Config: runs {P5_RUNS} | chunk {P5_CHUNK_LEN} tok, overlap {P5_OVERLAP}, {P5_CAP} chunks/doc ({P5_SELECT})"
      f" | head {P5_POOL}/{P5_LOSS} | FT minutes {P5_FT_MIN}")

In [ ]:
# P5-4. Label space, label statistics, and the chunk store
#   Each document -> sliding-window chunks (P5_CHUNK_LEN tokens, P5_OVERLAP overlap) -> a candidate pool of
#   the first P5_HEAD_CHUNKS, the last P5_TAIL_CHUNKS and the P5_KEY_CHUNKS chunks with the most statute
#   references ("section 302", "u/s", "IPC", "Article 21", "punishable", ...). Selection strategies then pick
#   P5_CAP chunks from the pool, so context-selection variants can be compared on the same cached features.
LABELS = sorted({l for r in LSI_TRAIN + LSI_DEV for l in r["labels"]})
LAB2COL = {l: j for j, l in enumerate(LABELS)}
NL = len(LABELS)

def to_multi_hot(records):
    Y = np.zeros((len(records), NL), dtype=np.float32)
    for i, r in enumerate(records):
        for l in r["labels"]:
            if l in LAB2COL: Y[i, LAB2COL[l]] = 1.0
    return Y

Y_TR, Y_DV = to_multi_hot(LSI_TRAIN), to_multi_hot(LSI_DEV)
TRAIN_FREQ = Y_TR.sum(0)
PRIOR = torch.tensor((TRAIN_FREQ + 0.5) / (len(Y_TR) + 1.0), dtype=torch.float32)
_q = np.quantile(TRAIN_FREQ[TRAIN_FREQ > 0], [1 / 3, 2 / 3]) if (TRAIN_FREQ > 0).any() else [1, 2]
FREQ_BUCKET = np.where(TRAIN_FREQ <= _q[0], "rare", np.where(TRAIN_FREQ <= _q[1], "medium", "frequent"))

print(f"✓ {NL} statute labels | train-only {int(((Y_TR.sum(0) > 0) & (Y_DV.sum(0) == 0)).sum())}, "
      f"dev-only {int(((Y_TR.sum(0) == 0) & (Y_DV.sum(0) > 0)).sum())}")
print(f"  label frequency in train: min {int(TRAIN_FREQ.min())}, median {int(np.median(TRAIN_FREQ))}, max {int(TRAIN_FREQ.max())}"
      f" | bucket cut-offs (docs) {np.round(_q, 1).tolist()}")
print(f"  docs with >1 statute: train {np.mean(Y_TR.sum(1) > 1):.1%}, dev {np.mean(Y_DV.sum(1) > 1):.1%}")

# ---- label co-occurrence (PMI) -----------------------------------------------
def cooccurrence_table(Y, top=15, min_count=5):
    C = Y.T @ Y; n = len(Y); f = np.diag(C)
    rows = []
    for a in range(NL):
        for b in range(a + 1, NL):
            if C[a, b] >= min_count:
                pmi = math.log(C[a, b] * n / (f[a] * f[b]))
                rows.append((LABELS[a], LABELS[b], int(C[a, b]), round(pmi, 2), round(C[a, b] / f[a], 2), round(C[a, b] / f[b], 2)))
    rows.sort(key=lambda r: -r[2])
    return rows[:top]

CO_TABLE = cooccurrence_table(Y_TR)
print("  most frequent statute pairs (a, b, docs, PMI, P(b|a), P(a|b)):", CO_TABLE[:6])

# ---- chunking -----------------------------------------------------------------
STATUTE_RE = re.compile(
    r"\b(?:sections?|sec|u\s*/\s*s|articles?|art|acts?|code|ipc|crpc|cpc|i\s*\.?\s*p\s*\.?\s*c|cr\s*\.?\s*p\s*\.?\s*c|"
    r"rules?|clauses?|sub\s*-?\s*sections?|schedule|constitution|punishable|offen[cs]es?|ordinance|provisions?)\b", re.I)
STATUTE_NUM_RE = re.compile(r"\b(?:sections?|sec|s|u\s*/\s*s|articles?|art|rules?)\s*\.?\s*\d+", re.I)

def statute_score(text):
    return len(STATUTE_RE.findall(text)) + 2 * len(STATUTE_NUM_RE.findall(text))

_WCACHE = {}
def _word_ids(w):
    v = _WCACHE.get(w)
    if v is None:
        v = list(tokenizer._word_ids(w)) if hasattr(tokenizer, "_word_ids") else list(tokenizer.encode_words([w]))
        _WCACHE[w] = v
    return v

def make_chunks(words, word_ids, L, overlap):
    """Greedy word-aligned windows of <= L tokens; consecutive windows share ~overlap tokens.
    Returns [(start_word, end_word, ids)]."""
    n, out, i = len(words), [], 0
    while i < n:
        ids, j = [], i
        while j < n and len(ids) + len(word_ids[j]) <= L:
            ids.extend(word_ids[j]); j += 1
        if j == i:                                   # a single word longer than L tokens
            ids, j = list(word_ids[i][:L]), i + 1
        out.append((i, j, ids))
        if j >= n: break
        k, back = j, 0
        while overlap > 0 and k > i + 1 and back < overlap:
            k -= 1; back += len(word_ids[k])
        i = k if overlap > 0 else j
    return out

def candidate_pool(chunks, scores, H, T, K):
    n = len(chunks)
    head = list(range(min(H, n))); tail = list(range(max(0, n - T), n))
    key = [c for c in sorted(range(n), key=lambda c: (-scores[c], c)) if scores[c] > 0][:K]
    return sorted(set(head) | set(tail) | set(key))

def build_doc(text, L=None, overlap=None):
    L = L or P5_CHUNK_LEN; overlap = P5_OVERLAP if overlap is None else overlap
    words = pretokenize(text, PREPROC) or ["."]
    wids = [_word_ids(w) or [PAD_ID] for w in words]
    chunks = make_chunks(words, wids, L, overlap)
    scores = [statute_score(" ".join(words[s:e])) for s, e, _ in chunks]
    pool = candidate_pool(chunks, scores, P5_HEAD_CHUNKS, P5_TAIL_CHUNKS, P5_KEY_CHUNKS)
    n = len(chunks)
    ids = np.full((len(pool), L), PAD_ID, dtype=np.int32); lens = np.zeros(len(pool), dtype=np.int16)
    meta = np.zeros((len(pool), 4), dtype=np.float32)
    for r, c in enumerate(pool):
        t = chunks[c][2][:L]; ids[r, :len(t)] = t; lens[r] = max(1, len(t))
        meta[r] = (c / max(1, n - 1), float(c < P5_HEAD_CHUNKS), float(c >= n - P5_TAIL_CHUNKS), math.log1p(scores[c]) / 3.0)
    return {"ids": ids, "lens": lens, "meta": meta, "n_chunks": n, "pos": np.array(pool, dtype=np.int32),
            "score": np.array([scores[c] for c in pool], dtype=np.float32)}

def select_chunks(doc, strategy, cap):
    """Indices into the doc's candidate pool, by priority, at most cap."""
    P = len(doc["pos"]); pos, sc, meta = doc["pos"], doc["score"], doc["meta"]
    if strategy == "all":
        order = list(range(P))
    elif strategy == "headtail":
        order = [i for i in range(P) if meta[i, 1] > 0] + [i for i in range(P) if meta[i, 2] > 0 and meta[i, 1] == 0]
    elif strategy == "keyword":
        first = [i for i in range(P) if pos[i] == 0]
        rest = sorted([i for i in range(P) if pos[i] != 0], key=lambda i: (-sc[i], -meta[i, 2], pos[i]))
        order = first + rest
    else:
        raise ValueError(strategy)
    cap = P if strategy == "all" else cap
    return sorted(order[:cap]) or [0]

class ChunkStore:
    def __init__(self, records, name):
        tag = hashlib.md5(json.dumps([name, len(records), records[0]["id"] if records else "", records[-1]["id"] if records else "",
                                      PREPROC, P5_CHUNK_LEN, P5_OVERLAP, P5_HEAD_CHUNKS, P5_TAIL_CHUNKS, P5_KEY_CHUNKS]).encode()).hexdigest()[:10]
        path = os.path.join(P5_WORK, f"store_{name}_{tag}.pkl")
        if os.path.exists(path):
            self.docs = pickle.load(open(path, "rb"))
        else:
            t0, self.docs = time.time(), []
            for i, r in enumerate(records):
                self.docs.append(build_doc(r["text"]))
                if (i + 1) % 2000 == 0: print(f"    {name}: {i + 1}/{len(records)} docs ({time.time() - t0:.0f}s)")
            pickle.dump(self.docs, open(path, "wb"), protocol=4)
        self.ids = [r["id"] for r in records]
        self.name = name
        nc = np.array([d["n_chunks"] for d in self.docs]); npool = np.array([len(d["pos"]) for d in self.docs])
        print(f"  ✓ {name}: {len(self.docs)} docs | chunks/doc median {int(np.median(nc))}, 90th {int(np.percentile(nc, 90))}"
              f" | pool/doc mean {npool.mean():.1f} | docs fully covered by {P5_CAP} chunks: {np.mean(nc <= P5_CAP):.0%}")

    def __len__(self): return len(self.docs)

    def selection(self, strategy, cap):
        return [select_chunks(d, strategy, cap) for d in self.docs]

print("Building chunk stores ...")
STORE = {"train": ChunkStore(LSI_TRAIN, "train"), "dev": ChunkStore(LSI_DEV, "dev"), "test": ChunkStore(LSI_TEST, "test")}
P5_POOL_MAX = max(len(d["pos"]) for s in STORE.values() for d in s.docs)
_ex = STORE["dev"].docs[0]
print(f"  example dev doc: {_ex['n_chunks']} chunks, pool positions {_ex['pos'].tolist()}, statute scores {_ex['score'].astype(int).tolist()}")

In [ ]:
# P5-5. Chunk encoder (frozen Phase-1 LM -> chunk vectors) and cached frozen features
#   chunk vector = [masked mean of the final hidden states, masked mean of the RMS-normalised middle block]
def p5_backbone(arch, keep_frac=None, name=None):
    """Phase-1 backbone, all weights frozen; optionally keep only the first keep_frac of its blocks."""
    bb = build_backbone(arch, name or f"Arch{arch} backbone")
    if keep_frac is not None:
        n_keep = max(1, int(round(len(bb.blocks) * keep_frac)))
        bb.blocks = nn.ModuleList(list(bb.blocks)[:n_keep])
        print(f"  Arch{arch}: kept the first {n_keep} blocks (student)")
    for p in bb.parameters(): p.requires_grad = False
    return bb.to(device).eval()

def _first(o):
    return o[0] if isinstance(o, (tuple, list)) else o

class ChunkEncoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        with torch.no_grad():
            ids = torch.full((1, 8), PAD_ID + 1, dtype=torch.long, device=device)
            self.out_dim = int(self(ids, torch.tensor([8], device=device)).shape[-1])

    def forward(self, ids, lens):
        blocks = self.backbone.blocks
        store = {}
        h_ = blocks[max(0, len(blocks) // 2 - 1)].register_forward_hook(lambda m, i, o: store.__setitem__("mid", _first(o)))
        try:
            h = _first(self.backbone(ids))
        finally:
            h_.remove()
        T = ids.shape[1]
        mf = (torch.arange(T, device=ids.device)[None, :] < lens[:, None]).unsqueeze(-1).float()
        n = mf.sum(1).clamp(min=1.0)
        a = (h.float() * mf).sum(1) / n
        m = store["mid"].float()
        m = m * torch.rsqrt(m.pow(2).mean(-1, keepdim=True) + 1e-6)
        return torch.cat([a, (m * mf).sum(1) / n], -1)

def collate_chunks(store, doc_idx, sel):
    """Stack the selected chunks of several documents. Returns ids, lens, meta (on device) and per-doc counts."""
    ids, lens, meta, counts = [], [], [], []
    for d in doc_idx:
        doc, s = store.docs[d], sel[d]
        ids.append(doc["ids"][s]); lens.append(doc["lens"][s]); meta.append(doc["meta"][s]); counts.append(len(s))
    ids = torch.from_numpy(np.concatenate(ids)).long()
    lens = torch.from_numpy(np.concatenate(lens).astype(np.int64))
    Tm = int(lens.max())                                  # trim common right padding
    return ids[:, :Tm].to(device), lens.to(device), torch.from_numpy(np.concatenate(meta)).to(device), counts

def pad_by_doc(x, counts):
    """[sumC, D] -> [B, Cmax, D] + mask [B, Cmax]."""
    B, Cm = len(counts), max(counts)
    out = x.new_zeros((B, Cm, x.shape[-1])); mask = torch.zeros((B, Cm), dtype=torch.bool, device=x.device)
    o = 0
    for b, c in enumerate(counts):
        out[b, :c] = x[o:o + c]; mask[b, :c] = True; o += c
    return out, mask

@torch.no_grad()
def check_amp(enc, key):
    """Use fp16 autocast only if it reproduces the fp32 chunk vectors."""
    if device.type != "cuda":
        P5_AMP_OK[key] = False; return False
    d = list(range(min(8, len(STORE["dev"]))))
    sel = [[0]] * len(STORE["dev"])
    ids, lens, _, _ = collate_chunks(STORE["dev"], d, sel)
    f32 = enc(ids, lens)
    with torch.autocast("cuda", dtype=AMP_DTYPE_P5):
        f16 = enc(ids, lens).float()
    cos = F.cosine_similarity(f32, f16, dim=-1).min().item()
    ok = bool(torch.isfinite(f16).all()) and cos > 0.995
    P5_AMP_OK[key] = ok
    print(f"  [{key}] fp16 autocast check: min cosine to fp32 = {cos:.4f} -> {'fp16' if ok else 'fp32'}")
    return ok

@torch.no_grad()
def extract_bank(enc, key, split, bs_chunks=48):
    """Frozen chunk vectors for every chunk in every document's pool: X [N, P, D] fp16, meta [N, P, 4], mask [N, P]."""
    store = STORE[split]
    path = os.path.join(P5_WORK, f"bank_{key}_{split}_{len(store)}_{hashlib.md5(store.ids[0].encode()).hexdigest()[:6] if len(store) else 0}_{P5_CHUNK_LEN}_{P5_POOL_MAX}_{PREPROC}.pt")
    if os.path.exists(path):
        return torch.load(path)
    t0, N, P = time.time(), len(store), P5_POOL_MAX
    X = torch.zeros((N, P, enc.out_dim), dtype=torch.float16); M = torch.zeros((N, P, 4)); K = torch.zeros((N, P), dtype=torch.bool)
    full = [list(range(len(d["pos"]))) for d in store.docs]
    order = sorted(range(N), key=lambda i: len(full[i]))
    amp = P5_AMP_OK.get(key, False)
    batch, nb = [], 0
    def flush(batch):
        ids, lens, meta, counts = collate_chunks(store, batch, full)
        with torch.autocast("cuda", dtype=AMP_DTYPE_P5, enabled=amp):
            f = enc(ids, lens).float().cpu()
        o = 0
        for d, c in zip(batch, counts):
            X[d, :c] = f[o:o + c].half(); M[d, :c] = meta[o:o + c].cpu(); K[d, :c] = True; o += c
    for d in order:
        batch.append(d); nb += len(full[d])
        if nb >= bs_chunks:
            flush(batch); batch, nb = [], 0
    if batch: flush(batch)
    bank = {"X": X, "meta": M, "mask": K}
    torch.save(bank, path)
    print(f"  ✓ frozen features {key}/{split}: {tuple(X.shape)} in {time.time() - t0:.0f}s")
    return bank

def sel_mask(split, strategy, cap):
    m = torch.zeros((len(STORE[split]), P5_POOL_MAX), dtype=torch.bool)
    for i, s in enumerate(STORE[split].selection(strategy, cap)):
        m[i, s] = True
    return m

@torch.no_grad()
def p5_sanity(bb, arch):
    if "lm_ppl" in globals() and "PPL_IDS" in globals():
        try:
            print(f"  [Arch{arch}] perplexity on Phase-4 sample (fp32): {lm_ppl(bb, PPL_IDS, use_amp=False):.1f}")
        except Exception as e:
            print(f"  [Arch{arch}] perplexity check skipped: {e}")
    n_par = sum(p.numel() for p in bb.parameters())
    print(f"  [Arch{arch}] {len(bb.blocks)} blocks, {n_par / 1e6:.1f}M parameters")
    return n_par

print("✓ Encoder utilities ready.")

In [ ]:
# P5-6. Multi-label head, losses, metrics and decision strategies
#   Head : fixed standardisation -> MLP per chunk -> label-wise attention over chunks (each statute attends to
#          the chunks that mention it) or mean+max pooling -> one logit per statute (binary relevance);
#          biases start at the log-odds of each statute's training frequency; optional label-correlation layer.
#   Loss : ASL (asymmetric loss, down-weights the flood of easy negatives), pos-weighted BCE, or plain BCE.
#   Decisions: one global threshold | per-statute thresholds (enough dev positives) | top-k; every document
#          gets at least its highest-scoring statute. Thresholds are honest-scored by 2-fold cross-fitting on dev.
from sklearn.metrics import f1_score

class LSIHead(nn.Module):
    def __init__(self, din, n_labels, prior, mu, sd, hid=None, pool=None, corr=None, drop=0.1):
        super().__init__()
        hid, pool, corr = hid or P5_HID, pool or P5_POOL, P5_CORR if corr is None else corr
        self.cfg = {"din": din, "n_labels": n_labels, "hid": hid, "pool": pool, "corr": corr}
        self.register_buffer("mu", mu.float().clone()); self.register_buffer("sd", sd.float().clamp(min=1e-4).clone())
        self.inp = nn.Sequential(nn.Dropout(drop), nn.Linear(din + 4, hid), nn.GELU(), nn.LayerNorm(hid), nn.Dropout(drop))
        b0 = torch.logit(prior.float().clamp(1e-4, 1 - 1e-4))
        self.pool = pool
        if pool == "lwan":
            self.U = nn.Parameter(torch.randn(n_labels, hid) / math.sqrt(hid))
            self.W = nn.Parameter(torch.randn(n_labels, hid) * 0.01)
            self.b = nn.Parameter(b0.clone())
        else:
            self.out = nn.Linear(2 * hid, n_labels)
            nn.init.normal_(self.out.weight, std=0.01)
            with torch.no_grad(): self.out.bias.copy_(b0)
        self.corr = None
        if corr:
            self.corr = nn.Linear(n_labels, n_labels, bias=False); nn.init.zeros_(self.corr.weight)
            self.register_buffer("offdiag", 1.0 - torch.eye(n_labels))

    def forward(self, x, meta, mask):
        x = (x.float() - self.mu) / self.sd
        h = self.inp(torch.cat([x, meta.float()], -1))                  # [B, C, hid]
        if self.pool == "lwan":
            s = torch.einsum("bch,lh->blc", h, self.U).masked_fill(~mask[:, None, :], -1e4)
            d = torch.einsum("blc,bch->blh", s.softmax(-1), h)
            z = (d * self.W).sum(-1) + self.b
        else:
            mf = mask.unsqueeze(-1).float()
            mean = (h * mf).sum(1) / mf.sum(1).clamp(min=1.0)
            mx = h.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values
            z = self.out(torch.cat([mean, mx], -1))
        if self.corr is not None:
            z = z + F.linear(torch.sigmoid(z), self.corr.weight * self.offdiag)
        return z

POS_WEIGHT = torch.tensor(np.clip(np.sqrt((len(Y_TR) - TRAIN_FREQ) / np.maximum(TRAIN_FREQ, 1)), 1.0, 30.0), dtype=torch.float32)

def asl_loss(z, y, gamma_neg=4.0, gamma_pos=0.0, clip=0.05):
    p = torch.sigmoid(z)
    p_neg = (p - clip).clamp(min=0) if clip > 0 else p
    lp = y * F.logsigmoid(z) * (1 - p).pow(gamma_pos)
    ln = (1 - y) * torch.log((1 - p_neg).clamp(min=1e-8)) * p_neg.pow(gamma_neg)
    return -(lp + ln).sum(-1).mean()

def task_loss(z, y, kind=None):
    kind = kind or P5_LOSS
    z, y = z.float(), y.float()
    if kind == "asl": return asl_loss(z, y)
    if kind == "wbce": return F.binary_cross_entropy_with_logits(z, y, pos_weight=POS_WEIGHT.to(z.device), reduction="none").sum(-1).mean()
    return F.binary_cross_entropy_with_logits(z, y, reduction="none").sum(-1).mean()

def kd_loss(z_s, z_t, T=None):
    """Per-statute binary KL(teacher_T || student_T) * T^2, summed over statutes, batch mean."""
    T = T or P5_KD_T
    pt = torch.sigmoid(z_t.float() / T)
    zs = z_s.float() / T
    ce = F.binary_cross_entropy_with_logits(zs, pt, reduction="none")
    ent = F.binary_cross_entropy(pt, pt, reduction="none")
    return (T ** 2) * (ce - ent).sum(-1).mean()

# ---------------------------- metrics & decisions (numpy) -----------------------
def lsi_metrics(Y, B):
    """Macro-F1 over the statutes present in the reference (as the evaluator derives the label set from it)."""
    Y, B = Y.astype(bool), B.astype(bool)
    present = Y.any(0)
    if not present.any(): return {"macro_f1": 0.0, "micro_f1": 0.0, "subset_acc": 0.0, "avg_pred_labels": 0.0}
    tp = (Y & B).sum(0); fp = (~Y & B).sum(0); fn = (Y & ~B).sum(0)
    f1 = 2 * tp[present] / (2 * tp[present] + fp[present] + fn[present])
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    return {"macro_f1": float(f1.mean()), "micro_f1": float(2 * TP / max(1, 2 * TP + FP + FN)),
            "subset_acc": float((Y == B).all(1).mean()), "avg_pred_labels": float(B.sum(1).mean())}

def per_label_f1(Y, B):
    Y, B = Y.astype(bool), B.astype(bool)
    tp = (Y & B).sum(0); fp = (~Y & B).sum(0); fn = (Y & ~B).sum(0)
    return np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan)

def decide(P, rule):
    kind, val = rule
    if kind == "topk":
        B = np.zeros_like(P, dtype=bool)
        idx = np.argsort(-P, axis=1)[:, :int(val)]
        np.put_along_axis(B, idx, True, axis=1)
        return B
    thr = np.asarray(val, dtype=np.float64)
    B = P >= thr
    empty = ~B.any(1)
    if empty.any():                                                      # at least one statute per document
        best = np.argmax(P[empty] - thr if thr.ndim else P[empty], axis=1)
        B[np.where(empty)[0], best] = True
    return B

GRID = np.round(np.concatenate([np.arange(0.02, 0.1, 0.01), np.arange(0.1, 0.951, 0.025)]), 3)

def fit_rule(P, Y, kind):
    if kind == "topk":
        best = max(range(1, 6), key=lambda k: lsi_metrics(Y, decide(P, ("topk", k)))["macro_f1"])
        return ("topk", best)
    g = max(GRID, key=lambda t: lsi_metrics(Y, decide(P, ("global", t)))["macro_f1"])
    if kind == "global":
        return ("global", float(g))
    thr = np.full(P.shape[1], g)
    Yb = Y.astype(bool); pos = Yb.sum(0)
    PB = P[:, :, None] >= GRID[None, None, :]                          # [N, L, G]
    tp = (PB & Yb[:, :, None]).sum(0); fp = (PB & ~Yb[:, :, None]).sum(0)
    f1 = 2 * tp / np.maximum(tp + fp + pos[:, None], 1)               # fn = pos - tp
    ok = pos >= P5_MIN_POS_THR
    thr[ok] = GRID[np.argmax(f1[ok], axis=1)]
    return ("per_label", thr)

def crossfit(P, Y, kind, seed=P5_SEED):
    """2-fold cross-fitted dev score of a decision strategy (thresholds never scored on the docs they were tuned on)."""
    idx = np.random.RandomState(seed).permutation(len(P)); a, b = idx[: len(P) // 2], idx[len(P) // 2:]
    out = []
    for tr, te in ((a, b), (b, a)):
        rule = fit_rule(P[tr], Y[tr], kind)
        out.append(lsi_metrics(Y[te], decide(P[te], rule)))
    return {k: float(np.mean([o[k] for o in out])) for k in out[0]}

def evaluate_probs(P, Y):
    """All decision strategies: cross-fitted score + rule fitted on the full dev set (used for test)."""
    res = {}
    for kind in ("global", "per_label", "topk"):
        rule = fit_rule(P, Y, kind)
        res[kind] = {"crossfit": crossfit(P, Y, kind), "insample": lsi_metrics(Y, decide(P, rule)), "rule": rule}
    best = max(res, key=lambda k: res[k]["crossfit"]["macro_f1"])
    return res, best

def quick_dev_f1(P, Y):
    return lsi_metrics(Y, decide(P, fit_rule(P, Y, "global")))["macro_f1"]

print("✓ Head, losses, metrics and decision strategies ready.")

In [ ]:
# P5-7. Stage A: train the head on cached frozen chunk features (fast; also used for ablations)
#        + ablations on the frozen Arch-4 features: context selection, pooling, loss, label correlation
RESULTS_PATH = os.path.join(P5_OUT, "p5_results.json")
P5_RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}

def save_results():
    json.dump(P5_RESULTS, open(RESULTS_PATH, "w"), indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))

def bank_stats(bank, max_rows=60000):
    idx = bank["mask"].nonzero()
    if len(idx) > max_rows: idx = idx[torch.randperm(len(idx), generator=torch.Generator().manual_seed(0))[:max_rows]]
    X = bank["X"][idx[:, 0], idx[:, 1]].float()
    return X.mean(0), X.std(0)

@torch.no_grad()
def head_logits(head, bank, smask, bs=128):
    head.eval(); out = []
    for s in range(0, bank["X"].shape[0], bs):
        x = bank["X"][s:s + bs].to(device); m = bank["meta"][s:s + bs].to(device)
        k = (bank["mask"][s:s + bs] & smask[s:s + bs]).to(device)
        out.append(head(x, m, k).float().cpu())
    return torch.cat(out).numpy()

def sigmoid_np(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def train_stage_a(bank_tr, bank_dv, sm_tr, sm_dv, cfg, epochs, desc, teacher_logits=None, verbose=True):
    p5_seed()
    mu, sd = bank_stats(bank_tr)
    head = LSIHead(bank_tr["X"].shape[-1], NL, PRIOR, mu, sd, pool=cfg["pool"], corr=cfg["corr"]).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=P5_A_LR, weight_decay=0.01)
    N, bs = bank_tr["X"].shape[0], P5_A_BS
    total = epochs * math.ceil(N / bs); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    Ytr = torch.from_numpy(Y_TR); Tz = None if teacher_logits is None else torch.as_tensor(teacher_logits, dtype=torch.float32)
    best, bad, t0 = (-1.0, None, None), 0, time.time()
    eval_every = 1 if epochs <= 10 else 2
    for ep in range(epochs):
        head.train()
        for b in torch.randperm(N).split(bs):
            x = bank_tr["X"][b].to(device); m = bank_tr["meta"][b].to(device); k = (bank_tr["mask"][b] & sm_tr[b]).to(device)
            z = head(x, m, k)
            loss = task_loss(z, Ytr[b].to(device), cfg["loss"])
            if Tz is not None:
                loss = P5_KD_ALPHA * kd_loss(z, Tz[b].to(device)) + (1 - P5_KD_ALPHA) * loss
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step(); sched.step()
        if (ep + 1) % eval_every == 0 or ep == epochs - 1:
            zd = head_logits(head, bank_dv, sm_dv)
            f1 = quick_dev_f1(sigmoid_np(zd), Y_DV)
            if f1 > best[0]:
                best, bad = (f1, {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}, zd), 0
            else:
                bad += 1
            if verbose: print(f"    [{desc}] epoch {ep + 1}/{epochs} loss {loss.item():.3f} dev macro-F1 {f1:.4f} (best {best[0]:.4f})")
            if bad >= 6: break
    head.load_state_dict(best[1])
    if verbose: print(f"  ✓ Stage A [{desc}] best dev macro-F1 (global thr, in-sample) {best[0]:.4f} in {time.time() - t0:.0f}s")
    return head, best[2]

def head_cfg():
    return {"select": P5_SELECT, "pool": P5_POOL, "loss": P5_LOSS, "corr": P5_CORR}

# ---- label-prior baseline (no model): every document gets the k most frequent statutes ----------------
_prior_P = np.tile(PRIOR.numpy()[None, :], (len(Y_DV), 1))
P5_RESULTS["baseline_prior_topk"] = {"dev": evaluate_probs(_prior_P + 1e-9 * np.random.rand(*_prior_P.shape), Y_DV)[0]["topk"]["crossfit"]}
print("Label-prior baseline (top-k most frequent statutes), dev:", P5_RESULTS["baseline_prior_topk"]["dev"])

# ---- ablations on frozen Arch-4 (or the first available architecture) features ----------------------
P5_ABL_ARCH = 4
if P5_ABLATION:
    free_mem()
    _bb = p5_backbone(P5_ABL_ARCH, name=f"Arch{P5_ABL_ARCH} (ablation, frozen)")
    p5_sanity(_bb, P5_ABL_ARCH)
    _enc = ChunkEncoder(_bb); check_amp(_enc, f"arch{P5_ABL_ARCH}")
    _btr = extract_bank(_enc, f"arch{P5_ABL_ARCH}", "train"); _bdv = extract_bank(_enc, f"arch{P5_ABL_ARCH}", "dev")
    del _enc, _bb; free_mem()
    base = head_cfg()
    grid = [dict(base, select=s) for s in ("headtail", "keyword", "all")]
    grid += [dict(base, pool=p) for p in ("meanmax", "lwan")]
    grid += [dict(base, loss=l) for l in ("bce", "wbce", "asl")]
    grid += [dict(base, corr=True)]
    seen, rows = set(), []
    for cfg in grid:
        sig = json.dumps(cfg, sort_keys=True)
        if sig in seen: continue
        seen.add(sig)
        smt, smd = sel_mask("train", cfg["select"], P5_CAP), sel_mask("dev", cfg["select"], P5_CAP)
        _, zd = train_stage_a(_btr, _bdv, smt, smd, cfg, P5_ABL_EPOCHS, f"abl {cfg}", verbose=False)
        res, bestk = evaluate_probs(sigmoid_np(zd), Y_DV)
        rows.append({**cfg, "strategy": bestk, **{f"dev_{k}": round(v, 4) for k, v in res[bestk]["crossfit"].items()}})
        print(f"  ablation {cfg} -> cross-fitted dev macro-F1 {res[bestk]['crossfit']['macro_f1']:.4f} ({bestk} thresholds)")
    P5_RESULTS["ablation"] = rows
    best_row = max([r for r in rows if r["select"] != "all"], key=lambda r: r["dev_macro_f1"])
    P5_SELECT, P5_POOL, P5_LOSS, P5_CORR = best_row["select"], best_row["pool"], best_row["loss"], best_row["corr"]
    P5_RESULTS["chosen_head_cfg"] = head_cfg(); save_results()
    try:
        import pandas as pd; display(pd.DataFrame(rows).sort_values("dev_macro_f1", ascending=False))
    except Exception:
        for r in rows: print(r)
    print(f"✓ Chosen for all runs: {head_cfg()} ('all' context is reported but not used: it costs "
          f"{P5_POOL_MAX} chunks/doc in fine-tuning)")
    del _btr, _bdv; free_mem()
elif "chosen_head_cfg" in P5_RESULTS:
    c = P5_RESULTS["chosen_head_cfg"]; P5_SELECT, P5_POOL, P5_LOSS, P5_CORR = c["select"], c["pool"], c["loss"], c["corr"]
    print("✓ Using the head configuration chosen in an earlier session:", c)

In [ ]:
# P5-8. Adaptation techniques (all start from the Phase-1 weights; every adapter starts as the identity)
#   LoRMA     : multiplicative low-rank update W' = (I + s·B·A) W on every Linear inside the blocks; B = 0 at init
#   Adapters  : Houlsby-style bottleneck after every block, x + up(GELU(down(x))); up = 0 at init
#   Layer-wise: unfreeze the top N blocks + final norm, learning rate decays by P5_LW_DECAY per block downwards
#   KD student: Arch 1 with only the first half of its blocks, fully fine-tuned on teacher + gold labels
class LoRMALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16.0, dropout=0.05):
        super().__init__()
        self.base = base
        out = base.out_features
        self.A = nn.Parameter(torch.randn(r, out, device=base.weight.device) / math.sqrt(out))
        self.B = nn.Parameter(torch.zeros(out, r, device=base.weight.device))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    @property
    def weight(self):           # merged weight, for any code that reads .weight directly
        return self.base.weight + self.scale * (self.B @ (self.A @ self.base.weight))

    @property
    def bias(self): return self.base.bias

    @property
    def in_features(self): return self.base.in_features

    @property
    def out_features(self): return self.base.out_features

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + self.scale * F.linear(F.linear(self.drop(h), self.A.to(h.dtype)), self.B.to(h.dtype))
        return h if self.base.bias is None else h + self.base.bias.to(h.dtype)

def inject_lorma(bb, r=None, alpha=None, dropout=0.05):
    targets = []
    for blk in bb.blocks:
        for name, mod in blk.named_modules():
            if isinstance(mod, nn.Linear) and not isinstance(mod, LoRMALinear) and ".base" not in f".{name}":
                targets.append((blk, name))
    for blk, name in targets:
        parent = blk.get_submodule(name.rsplit(".", 1)[0]) if "." in name else blk
        child = name.rsplit(".", 1)[-1]
        setattr(parent, child, LoRMALinear(getattr(parent, child), r or P5_LORMA_R, alpha or P5_LORMA_ALPHA, dropout))
    print(f"  LoRMA injected into {len(targets)} Linear layers")
    return [p for m in bb.modules() if isinstance(m, LoRMALinear) for p in (m.A, m.B)]

class AdapterBlock(nn.Module):
    def __init__(self, block, d_model, dim):
        super().__init__()
        self.block = block
        self.down, self.up = nn.Linear(d_model, dim), nn.Linear(dim, d_model)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        out = self.block(*a, **k)
        x = out[0] if isinstance(out, (tuple, list)) else out
        x = x + self.up(F.gelu(self.down(x)))
        return (x, *out[1:]) if isinstance(out, tuple) else ([x, *out[1:]] if isinstance(out, list) else x)

    def __getattr__(self, name):            # keep block.attn / block.ffn etc. reachable
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["block"], name)

def d_model_of(bb):
    emb = [m for m in bb.modules() if isinstance(m, nn.Embedding)]
    return int(max(emb, key=lambda e: e.num_embeddings).embedding_dim)

def inject_adapters(bb, dim=None):
    D = d_model_of(bb)
    bb.blocks = nn.ModuleList([AdapterBlock(b, D, dim or P5_ADAPTER_DIM).to(device) for b in bb.blocks])
    print(f"  bottleneck adapters ({D} -> {dim or P5_ADAPTER_DIM} -> {D}) after {len(bb.blocks)} blocks")
    return [p for b in bb.blocks for p in list(b.down.parameters()) + list(b.up.parameters())]

def final_norm(bb):
    for n in ("ln_f", "norm_f", "final_norm", "norm", "ln_final", "final_layernorm"):
        m = getattr(bb, n, None)
        if isinstance(m, nn.Module): return m
    return None

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr

print("✓ LoRMA, adapters and layer-wise utilities ready.")

In [ ]:
# P5-9. Stage B (end-to-end adaptation) engine, prediction, cost accounting, submission writing
#   run_experiment: frozen features -> Stage A head (recorded as the "<run>_frozen" member: how useful are the
#   raw Phase-1 representations?) -> inject the adaptation technique (identity at init, so Stage B starts at the
#   Stage-A solution) -> time-budgeted fine-tuning with dev-based model selection -> thresholds -> submission.
class DocModel(nn.Module):
    def __init__(self, enc, head):
        super().__init__()
        self.enc, self.head = enc, head

    def forward(self, ids, lens, meta, counts, amp):
        with torch.autocast(device.type, dtype=AMP_DTYPE_P5, enabled=amp):
            f = self.enc(ids, lens)
        x, mask = pad_by_doc(f.float(), counts)
        m, _ = pad_by_doc(meta.float(), counts)
        with torch.autocast(device.type, enabled=False):
            return self.head(x, m, mask)

@torch.no_grad()
def predict_model(model, split, sel, amp, docs_per_batch=8):
    model.eval(); store = STORE[split]; out = []
    for s in range(0, len(store), docs_per_batch):
        b = list(range(s, min(s + docs_per_batch, len(store))))
        ids, lens, meta, counts = collate_chunks(store, b, sel)
        out.append(model(ids, lens, meta, counts, amp).float().cpu())
    return torch.cat(out).numpy()

def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def finetune(model, key, groups, minutes, amp, sel_tr, sel_dv, backbone_train=False, teacher=None):
    opt = torch.optim.AdamW(groups)
    base_lrs = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=amp and device.type == "cuda")
    Ytr = torch.from_numpy(Y_TR); Tz = None if teacher is None else torch.as_tensor(teacher, dtype=torch.float32)
    budget, dpf = minutes * 60.0, P5_DOCS_PER_FWD
    t0 = time.time(); next_eval = P5_EVAL_EVERY_MIN * 60.0
    best_f1 = quick_dev_f1(sigmoid_np(predict_model(model, "dev", sel_dv, amp)), Y_DV)
    best_state, step, audited, losses = trainable_state(model), 0, False, []
    print(f"  [{key}] Stage B start: dev macro-F1 {best_f1:.4f} | budget {minutes} min | "
          f"trainable {count_params(model)[1] / 1e6:.2f}M params")
    N = len(STORE["train"]); done = budget <= 0
    while not done:
        order = np.random.permutation(N).tolist()
        for i in range(0, N, dpf * P5_ACCUM):
            el = time.time() - t0
            if el >= budget: done = True; break
            frac = el / budget
            f = min(1.0, frac / 0.05) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * frac)))
            for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * f
            model.train()
            if not backbone_train: model.enc.backbone.eval()
            opt.zero_grad(set_to_none=True)
            chunk = order[i:i + dpf * P5_ACCUM]; micro = [chunk[j:j + dpf] for j in range(0, len(chunk), dpf)]
            try:
                for mb in micro:
                    ids, lens, meta, counts = collate_chunks(STORE["train"], mb, sel_tr)
                    z = model(ids, lens, meta, counts, amp)
                    loss = task_loss(z, Ytr[mb].to(device))
                    if Tz is not None:
                        loss = P5_KD_ALPHA * kd_loss(z, Tz[mb].to(device)) + (1 - P5_KD_ALPHA) * loss
                    scaler.scale(loss / len(micro)).backward()
                    losses.append(loss.item())
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True); free_mem()
                dpf = max(1, dpf // 2)
                print(f"    OOM -> {dpf} documents per forward pass"); continue
            if not audited:
                tr = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
                nz = sum(1 for _, p in tr if p.grad is not None and p.grad.abs().sum() > 0)
                print(f"    gradient audit: {nz}/{len(tr)} trainable tensors receive gradient")
                audited = True
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            el = time.time() - t0
            if el >= next_eval or el >= budget:
                f1 = quick_dev_f1(sigmoid_np(predict_model(model, "dev", sel_dv, amp)), Y_DV)
                tag = ""
                if f1 > best_f1: best_f1, best_state, tag = f1, trainable_state(model), "  *best*"
                print(f"    [{key}] {el / 60:5.1f} min | step {step} | loss {np.mean(losses[-50:]):.3f} | dev macro-F1 {f1:.4f}{tag}")
                next_eval += P5_EVAL_EVERY_MIN * 60.0
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in best_state: p.copy_(best_state[n].to(p.device))
    print(f"  ✓ [{key}] Stage B done: {step} steps, best dev macro-F1 {best_f1:.4f}")
    return best_f1

def finalize_member(key, label, zd, zt, info, submit=True):
    Pd, Pt = sigmoid_np(zd), sigmoid_np(zt)
    res, best = evaluate_probs(Pd, Y_DV)
    np.save(os.path.join(P5_OUT, f"{key}_dev_logits.npy"), zd.astype(np.float32))
    np.save(os.path.join(P5_OUT, f"{key}_test_logits.npy"), zt.astype(np.float32))
    rule = res[best]["rule"]
    entry = {"label": label, "strategy": best, "rule": [rule[0], np.asarray(rule[1]).tolist()],
             "dev_crossfit": res[best]["crossfit"], "dev_insample": res[best]["insample"],
             "by_strategy": {k: {"crossfit": v["crossfit"], "insample": v["insample"]} for k, v in res.items()}, **info}
    if submit:
        B = decide(Pt, rule)
        preds = [{"id": i, "pred": sorted({int(LABELS[j]) for j in np.flatnonzero(row)})} for i, row in zip(STORE["test"].ids, B)]
        assert len(preds) == len(LSI_TEST) and all(p["pred"] and all(isinstance(x, int) for x in p["pred"]) for p in preds)
        assert all(len(p["pred"]) == len(set(p["pred"])) for p in preds)
        suffix = "_DRY" if P5_DRY_RUN else ""
        jpath = os.path.join(P5_OUT, f"{key}{suffix}_{P5_SUB_NAME}")
        json.dump(preds, open(jpath, "w"))
        zpath = os.path.join(P5_OUT, f"submission_{key}{suffix}.zip")
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(jpath, arcname=P5_SUB_NAME)
        entry["submission"] = zpath
        print(f"  ✓ submission written: {zpath} ({np.mean([len(p['pred']) for p in preds]):.2f} statutes/doc on test)")
    P5_RESULTS[key] = entry; save_results()
    cf = res[best]["crossfit"]
    print(f"  ✓ {label}: dev (cross-fitted, {best}) macro-F1 {cf['macro_f1']:.4f} | micro-F1 {cf['micro_f1']:.4f}"
          f" | subset acc {cf['subset_acc']:.4f}")
    return entry

def run_experiment(key, label, arch, technique, minutes, keep_frac=None, teacher_logits=None):
    free_mem(); p5_seed(); t_start = time.time()
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    print(f"\n{'=' * 90}\n{label}\n{'=' * 90}")
    bb = p5_backbone(arch, keep_frac); p5_sanity(bb, arch)
    enc = ChunkEncoder(bb); fkey = f"arch{arch}" + (f"_keep{keep_frac}" if keep_frac else "")
    amp = check_amp(enc, fkey)
    banks = {s: extract_bank(enc, fkey, s) for s in ("train", "dev", "test")}
    sel = {s: STORE[s].selection(P5_SELECT, P5_CAP) for s in STORE}
    sm = {s: sel_mask(s, P5_SELECT, P5_CAP) for s in STORE}
    head, zd_A = train_stage_a(banks["train"], banks["dev"], sm["train"], sm["dev"], head_cfg(), P5_A_EPOCHS,
                               f"{key} stage A", teacher_logits)
    zt_A = head_logits(head, banks["test"], sm["test"])
    t_A = time.time() - t_start
    finalize_member(f"{key}_frozen", f"{label} — frozen Phase-1 features + head (Stage A only)", zd_A, zt_A,
                    {"arch": arch, "technique": "frozen", "train_seconds": t_A,
                     "trainable_params": sum(p.numel() for p in head.parameters())}, submit=False)
    del banks; free_mem()

    head_group = {"params": list(head.parameters()), "lr": P5_HEAD_LR, "weight_decay": 0.01}
    backbone_train = False
    if technique == "lorma":
        groups = [{"params": inject_lorma(bb), "lr": P5_LORMA_LR, "weight_decay": 0.0}, head_group]
    elif technique == "adapters":
        groups = [{"params": inject_adapters(bb), "lr": P5_ADAPTER_LR, "weight_decay": 0.0}, head_group]
    elif technique == "layerwise":
        groups, nb = [head_group], len(bb.blocks)
        for d in range(min(P5_LW_BLOCKS, nb)):
            blk = bb.blocks[nb - 1 - d]
            for p in blk.parameters(): p.requires_grad = True
            groups.append({"params": list(blk.parameters()), "lr": P5_LW_TOP_LR * P5_LW_DECAY ** d, "weight_decay": 0.01})
        fn = final_norm(bb)
        if fn is not None:
            for p in fn.parameters(): p.requires_grad = True
            groups.append({"params": list(fn.parameters()), "lr": P5_LW_TOP_LR, "weight_decay": 0.0})
        backbone_train = True
        _lrs = ["%.1e" % g["lr"] for g in groups[1:]]
        print(f"  layer-wise: top {min(P5_LW_BLOCKS, nb)} of {nb} blocks unfrozen, LRs {_lrs}")
    elif technique == "student_full":
        for p in bb.parameters(): p.requires_grad = True
        groups = [{"params": list(bb.parameters()), "lr": P5_STUDENT_LR, "weight_decay": 0.01}, head_group]
        backbone_train = True
    else:
        raise ValueError(technique)
    model = DocModel(enc, head).to(device)
    finetune(model, key, groups, minutes, amp, sel["train"], sel["dev"], backbone_train, teacher_logits)
    train_s = time.time() - t_start
    peak = torch.cuda.max_memory_allocated() / 2 ** 20 if device.type == "cuda" else float("nan")
    zd = predict_model(model, "dev", sel["dev"], amp)
    t1 = time.time(); zt = predict_model(model, "test", sel["test"], amp); lat = (time.time() - t1) / max(1, len(LSI_TEST))
    tot, tr = count_params(model)
    info = {"arch": arch, "technique": technique, "total_params": tot, "trainable_params": tr,
            "model_mb_fp32": tot * 4 / 2 ** 20, "backbone_blocks": len(bb.blocks), "peak_train_mb": peak,
            "train_seconds": train_s, "latency_s_per_doc": lat, "amp": amp, "chunks_per_doc": P5_CAP, "select": P5_SELECT}
    finalize_member(key, label, zd, zt, info, submit=True)
    return model, sel, amp

print("✓ Stage-B engine ready.")

In [ ]:
# RUN 1 — Arch 4 (ALiBi + SwiGLU + RMSNorm) + LoRMA                      [compulsory LoRMA run; KD teacher]
#   After training, its logits on every training document are cached as the soft targets for Run 4.
TEACHER_PATH = os.path.join(P5_OUT, "teacher_train_logits.npy")

if 1 in P5_RUNS:
    model1, sel1, amp1 = run_experiment("LSI1_Arch4_LoRMA", "Run 1 — Arch 4 + LoRMA", 4, "lorma", P5_FT_MIN[1])
    zt_train = predict_model(model1, "train", sel1["train"], amp1)
    np.save(TEACHER_PATH, zt_train.astype(np.float32))
    json.dump(STORE["train"].ids, open(TEACHER_PATH.replace(".npy", "_ids.json"), "w"))
    print(f"✓ teacher logits cached for {len(zt_train)} training documents "
          f"(teacher train macro-F1 at 0.5: {lsi_metrics(Y_TR, decide(sigmoid_np(zt_train), ('global', 0.5)))['macro_f1']:.3f})")
    del model1, zt_train; free_mem()

In [ ]:
# RUN 2 — Arch 3 (GQA + learned positions + LayerNorm) + bottleneck Adapters
if 2 in P5_RUNS:
    model2, _, _ = run_experiment("LSI2_Arch3_Adapters", "Run 2 — Arch 3 + Adapters", 3, "adapters", P5_FT_MIN[2])
    del model2; free_mem()

In [ ]:
# RUN 3 — Arch 2 (RoPE + SwiGLU + RMSNorm) + partial / layer-wise fine-tuning
if 3 in P5_RUNS:
    model3, _, _ = run_experiment("LSI3_Arch2_Layerwise", "Run 3 — Arch 2 + layer-wise fine-tuning", 2, "layerwise", P5_FT_MIN[3])
    del model3; free_mem()

In [ ]:
# RUN 4 — Knowledge distillation: Run-1 model (Arch 4 + LoRMA, teacher) -> Arch-1 student with half its blocks
#   Student keeps the first P5_STUDENT_KEEP of Arch 1's Phase-1 blocks (smaller and faster than the teacher)
#   and is fully fine-tuned with  alpha * T^2 * KL(teacher_T || student_T) per statute + (1 - alpha) * task loss,
#   both in Stage A (frozen student features) and in Stage B (end-to-end).
def find_teacher():
    for p in [TEACHER_PATH] + sorted(glob.glob("/kaggle/input/**/teacher_train_logits.npy", recursive=True)):
        ip = p.replace(".npy", "_ids.json")
        if os.path.exists(p) and os.path.exists(ip):
            ids = json.load(open(ip)); z = np.load(p)
            pos = {i: k for k, i in enumerate(ids)}
            if all(i in pos for i in STORE["train"].ids) and z.shape[1] == NL:
                print("✓ teacher logits:", p)
                return z[[pos[i] for i in STORE["train"].ids]]
            print(f"  ⚠ {p} does not match the current training set / label space, skipped")
    return None

if 4 in P5_RUNS:
    T_LOGITS = find_teacher()
    assert T_LOGITS is not None, "Run 1 first (or attach the output that contains teacher_train_logits.npy)"
    model4, _, _ = run_experiment("LSI4_KD_Arch4_to_Arch1half", "Run 4 — KD: Arch 4 + LoRMA teacher -> half-depth Arch 1 student",
                                  1, "student_full", P5_FT_MIN[4], keep_frac=P5_STUDENT_KEEP, teacher_logits=T_LOGITS)
    del model4, T_LOGITS; free_mem()

In [ ]:
# P5-14. Comparison, optional ensemble, and the analysis asked for in 11.5
#   All dev numbers are 2-fold cross-fitted (thresholds tuned on one half of dev, scored on the other).
import pandas as pd
import matplotlib.pyplot as plt

P5_WRITE_ENSEMBLE = True
RUN_KEYS = ["LSI1_Arch4_LoRMA", "LSI2_Arch3_Adapters", "LSI3_Arch2_Layerwise", "LSI4_KD_Arch4_to_Arch1half"]

for p in sorted(glob.glob("/kaggle/input/**/p5_results.json", recursive=True)):       # earlier sessions
    for k, v in json.load(open(p)).items():
        P5_RESULTS.setdefault(k, v)

def load_logits(key, split):
    for d in [P5_OUT] + sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{key}_{split}_logits.npy", recursive=True)}):
        p = os.path.join(d, f"{key}_{split}_logits.npy")
        if os.path.exists(p): return np.load(p)
    return None

def crossfit_binary(P, Y, kind, seed=P5_SEED):
    idx = np.random.RandomState(seed).permutation(len(P)); a, b = idx[: len(P) // 2], idx[len(P) // 2:]
    B = np.zeros_like(P, dtype=bool)
    for tr, te in ((a, b), (b, a)):
        B[te] = decide(P[te], fit_rule(P[tr], Y[tr], kind))
    return B

AVAIL = {k: (load_logits(k, "dev"), load_logits(k, "test")) for k in RUN_KEYS}
AVAIL = {k: v for k, v in AVAIL.items() if v[0] is not None and v[0].shape == Y_DV.shape}
print("runs with saved predictions:", list(AVAIL))

# ---- ensemble of the four architectures --------------------------------
if len(AVAIL) >= 2:
    Pd_ens = np.mean([sigmoid_np(v[0]) for v in AVAIL.values()], 0)
    Pt_ens = np.mean([sigmoid_np(v[1]) for v in AVAIL.values()], 0)
    logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    finalize_member("LSI5_Ensemble", f"Ensemble (mean probability of {len(AVAIL)} runs)", logit(Pd_ens), logit(Pt_ens),
                    {"technique": "ensemble", "members": list(AVAIL)}, submit=P5_WRITE_ENSEMBLE)
    AVAIL["LSI5_Ensemble"] = (logit(Pd_ens), logit(Pt_ens))

# ---- 1. main comparison table --------------------------------------------------
rows = []
for k, e in P5_RESULTS.items():
    if not isinstance(e, dict) or "dev_crossfit" not in e: continue
    cf = e["dev_crossfit"]
    rows.append({"member": k, "technique": e.get("technique"), "arch": e.get("arch"), "decision": e.get("strategy"),
                 "dev_macro_f1": cf["macro_f1"], "dev_micro_f1": cf["micro_f1"], "dev_subset_acc": cf["subset_acc"],
                 "total_params_M": (e.get("total_params") or np.nan) / 1e6, "trainable_params_M": (e.get("trainable_params") or np.nan) / 1e6,
                 "model_MB_fp32": e.get("model_mb_fp32"), "peak_train_MB": e.get("peak_train_mb"),
                 "train_min": (e.get("train_seconds") or np.nan) / 60, "ms_per_doc": (e.get("latency_s_per_doc") or np.nan) * 1000})
if "baseline_prior_topk" in P5_RESULTS:
    b = P5_RESULTS["baseline_prior_topk"]["dev"]
    rows.append({"member": "baseline: most frequent statutes", "technique": "none", "dev_macro_f1": b["macro_f1"],
                 "dev_micro_f1": b["micro_f1"], "dev_subset_acc": b["subset_acc"]})
TABLE = pd.DataFrame(rows).sort_values("dev_macro_f1", ascending=False)
print("\n=== 1. Experiments (dev, cross-fitted) ==="); display(TABLE.round(4))

# ---- 2. Phase-1 representations: frozen vs adapted ---------------------------------
print("\n=== 2. Are the Phase-1 representations useful? frozen features + head vs adapted ===")
fz = [{"run": k, "frozen_dev_macro_f1": P5_RESULTS.get(f"{k}_frozen", {}).get("dev_crossfit", {}).get("macro_f1"),
       "adapted_dev_macro_f1": P5_RESULTS.get(k, {}).get("dev_crossfit", {}).get("macro_f1")} for k in RUN_KEYS]
display(pd.DataFrame(fz).round(4))

# ---- 3. decision strategies --------------------------------------------------------
print("\n=== 3. Threshold strategies (dev macro-F1: cross-fitted / in-sample) ===")
ts = []
for k in list(AVAIL):
    e = P5_RESULTS.get(k, {})
    for s, v in e.get("by_strategy", {}).items():
        ts.append({"run": k, "strategy": s, "crossfit_macro_f1": v["crossfit"]["macro_f1"], "insample_macro_f1": v["insample"]["macro_f1"],
                   "crossfit_micro_f1": v["crossfit"]["micro_f1"], "avg_labels_pred": v["crossfit"].get("avg_pred_labels")})
if ts: display(pd.DataFrame(ts).pivot_table(index="run", columns="strategy", values=["crossfit_macro_f1", "insample_macro_f1"]).round(4))
print(f"  gold statutes per dev document: {Y_DV.sum(1).mean():.2f}")

# ---- 4. context selection / pooling / loss / label correlation ablations ---------------
if P5_RESULTS.get("ablation"):
    print("\n=== 4. Ablations on frozen Arch-4 features (cross-fitted dev) ===")
    display(pd.DataFrame(P5_RESULTS["ablation"]).sort_values("dev_macro_f1", ascending=False))

# ---- 5. rare statutes, missed statutes, confusions -----------------------------------
best_key = max(AVAIL, key=lambda k: P5_RESULTS[k]["dev_crossfit"]["macro_f1"]) if AVAIL else None
if best_key:
    bucket_rows = []
    for k, (zd, _) in AVAIL.items():
        Bk = crossfit_binary(sigmoid_np(zd), Y_DV, P5_RESULTS[k]["strategy"])
        f1k = per_label_f1(Y_DV, Bk)
        for bname in ("rare", "medium", "frequent"):
            m = (FREQ_BUCKET == bname) & (Y_DV.sum(0) > 0)
            bucket_rows.append({"run": k, "bucket": bname, "n_statutes": int(m.sum()), "mean_F1": float(np.nanmean(f1k[m])) if m.any() else np.nan})
    print("\n=== 5a. F1 by statute frequency bucket (train docs per statute) ===")
    display(pd.DataFrame(bucket_rows).pivot_table(index="run", columns="bucket", values="mean_F1").round(3))

    P = sigmoid_np(AVAIL[best_key][0]); B = crossfit_binary(P, Y_DV, P5_RESULTS[best_key]["strategy"])
    Yb = Y_DV.astype(bool); f1 = per_label_f1(Y_DV, B)
    rec = np.where(Yb.sum(0) > 0, (Yb & B).sum(0) / np.maximum(Yb.sum(0), 1), np.nan)
    prec = np.where(B.sum(0) > 0, (Yb & B).sum(0) / np.maximum(B.sum(0), 1), np.nan)
    lab = pd.DataFrame({"statute": LABELS, "train_docs": TRAIN_FREQ.astype(int), "dev_docs": Yb.sum(0), "pred_docs": B.sum(0),
                        "precision": prec, "recall": rec, "F1": f1, "bucket": FREQ_BUCKET})
    print(f"\n=== 5b. Most-missed statutes for {best_key} (dev docs >= 3, lowest recall) ===")
    display(lab[lab.dev_docs >= 3].sort_values("recall").head(12).round(3))
    print(f"  statutes in dev never predicted: {int(((Yb.sum(0) > 0) & (B.sum(0) == 0)).sum())} of {int((Yb.sum(0) > 0).sum())}")
    conf = Counter()
    for i in range(len(Yb)):
        fps, trues = np.flatnonzero(B[i] & ~Yb[i]), np.flatnonzero(Yb[i] & ~B[i])
        for a in trues:
            for f in fps: conf[(LABELS[a], LABELS[f])] += 1
    print("\n=== 5c. Commonly confused statutes (missed gold -> wrongly predicted, same document) ===")
    display(pd.DataFrame([{"gold_missed": a, "predicted_instead": b, "docs": c} for (a, b), c in conf.most_common(15)]))

    # ---- 6. label co-occurrence -------------------------------------------------------
    print("\n=== 6. Label co-occurrence in train (a, b, docs, PMI, P(b|a), P(a|b)) ===")
    display(pd.DataFrame(CO_TABLE, columns=["a", "b", "docs", "PMI", "P(b|a)", "P(a|b)"]))
    pair_rows = []
    for a, b_, *_ in CO_TABLE[:10]:
        ja, jb = LAB2COL[a], LAB2COL[b_]
        both = Yb[:, ja] & Yb[:, jb]
        if both.sum(): pair_rows.append({"pair": f"{a}+{b_}", "dev_docs_with_both": int(both.sum()),
                                         "both_predicted": float((B[both, ja] & B[both, jb]).mean())})
    if pair_rows:
        print("  how often the best run recovers BOTH statutes of a frequent pair:"); display(pd.DataFrame(pair_rows).round(3))

    # ---- 7. plots ---------------------------------------------------------------------
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    m = Yb.sum(0) > 0
    ax[0].scatter(np.log10(TRAIN_FREQ[m] + 1), f1[m], s=12, alpha=0.6)
    ax[0].set_xlabel("log10(train documents per statute + 1)"); ax[0].set_ylabel("dev F1"); ax[0].set_title(f"{best_key}: per-statute F1 vs frequency")
    pr = [(t, *[lsi_metrics(Y_DV, decide(P, ("global", t)))[k] for k in ("macro_f1", "micro_f1")]) for t in GRID]
    prec_g = [((P >= t) & Yb).sum() / max(1, (P >= t).sum()) for t in GRID]; rec_g = [((P >= t) & Yb).sum() / Yb.sum() for t in GRID]
    ax[1].plot(GRID, [r[1] for r in pr], label="macro-F1"); ax[1].plot(GRID, [r[2] for r in pr], label="micro-F1")
    ax[1].plot(GRID, prec_g, "--", label="precision (micro)"); ax[1].plot(GRID, rec_g, "--", label="recall (micro)")
    ax[1].set_xlabel("global threshold"); ax[1].legend(); ax[1].set_title("precision / recall trade-off (in-sample)")
    plt.tight_layout(); plt.savefig(os.path.join(P5_OUT, "analysis_plots.png"), dpi=120); plt.show()

# ---- 8. KD: size / cost vs quality ---------------------------------------------------
t, s = P5_RESULTS.get("LSI1_Arch4_LoRMA"), P5_RESULTS.get("LSI4_KD_Arch4_to_Arch1half")
if t and s:
    print("\n=== 8. Knowledge distillation: teacher vs student ===")
    display(pd.DataFrame([{"model": n, "dev_macro_f1": e["dev_crossfit"]["macro_f1"], "params_M": e["total_params"] / 1e6,
                           "blocks": e.get("backbone_blocks"), "ms_per_doc": e["latency_s_per_doc"] * 1000, "peak_train_MB": e["peak_train_mb"]}
                          for n, e in (("teacher (Run 1)", t), ("student (Run 4)", s))]).round(3))

TABLE.to_csv(os.path.join(P5_OUT, "p5_comparison.csv"), index=False); save_results()
print("\nSubmissions:", sorted(glob.glob(os.path.join(P5_OUT, "submission_*.zip"))))